# NB1 — Aquisição, Validação e Pré-processamento de Contextos de Crise

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

Este notebook:
1. Testa conexão com cada fonte de dados (PhysioNet, OpenNeuro S3, Mendeley API)
2. Descobre pacientes e baixa arquivos de anotação da nuvem
3. Lê duração real de cada EDF via header remoto (Range request — sem baixar o arquivo)
4. Valida contextos de crise (interictal + pré-ictal) com todos os tempos em segundos
5. Baixa apenas os EDFs necessários para os contextos selecionados
6. Filtra, reamostra (256 Hz) e salva `.npz` por contexto
7. Exporta `mapa_contextos_nb1.json` e `pipeline_manifest.json`

**Fontes de dados:**
| Dataset  | Fonte                          | Acesso              |
|----------|--------------------------------|----------------------|
| CHB-MIT  | PhysioNet (HTTP)               | open                |
| Siena    | PhysioNet (HTTP)               | open                |
| SeizeIT2 | OpenNeuro `ds005873` (S3 anônimo) | open             |
| Mendeley | Mendeley Data API (`cloudscraper`) | open            |

**Parâmetros de contexto:**
- `GUARD = 30 min` — zona proibida pós-crise
- `MAX_PRE = 30 min` — janela pré-ictal máxima
- `MIN_INTER = 30 min` — interictal mínimo
- `MIN_CONTEXTS = 2` — contextos válidos mínimos por paciente

**Notas Siena:**
- PN06: txt de anotação usa `PNO6` (letra O); servidor PhysioNet usa `pn06` (zero). `normalize_fn()` corrige automaticamente.
- PN08: EDF corrompido — excluído.
- PN010: pasta com nome incorreto no disco (mesmo paciente que PN10) — excluído.


In [1]:
%pip install -q boto3 cloudscraper mne PyWavelets scipy tqdm openpyxl
print('Dependências OK.')


Note: you may need to restart the kernel to use updated packages.
Dependências OK.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, json, warnings, struct, io
from pathlib import Path
from collections import defaultdict
import urllib.request, urllib.error, urllib.parse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import boto3
from botocore import UNSIGNED
from botocore.config import Config

try:
    import cloudscraper
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'cloudscraper', '-q'])
    import cloudscraper

warnings.filterwarnings('ignore', category=RuntimeWarning)

try:
    import mne
    mne.set_log_level('ERROR')
    HAS_MNE = True
except ImportError:
    HAS_MNE = False
    print('MNE nao instalado — instale com: pip install mne')

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(it, **kw): return it

# ── Diretórios locais (cache dos arquivos baixados) ───────────────────────────
ROOT         = Path('data')
CHBMIT_DIR   = ROOT / 'chb-mit'
SIENA_DIR    = ROOT / 'siena'
SEIZEIT_DIR  = ROOT / 'seizeit2'
MENDELEY_DIR = ROOT / 'mendeley'
SIGNAL_DIR   = ROOT / 'signals'
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)

SEIZEIT_SESSION = 'ses-01'

# ── Parâmetros de contexto ────────────────────────────────────────────────────
GUARD_S      = 30 * 60
MAX_PRE_S    = 30 * 60
MIN_INTER_S  = 30 * 60
MIN_CONTEXTS = 2

# ── Parâmetros de pré-processamento ──────────────────────────────────────────
TARGET_SFREQ     = 256.0
F_HIGHPASS       = 0.5
F_LOWPASS        = 40.0
NOTCH_FREQS      = [50, 60]
MAX_INTER_SAVE_S = 150 * 60   # cap interictal salvo no .npz (evita OOM)
CLIP_UV          = 500.0
DATASETS_CAR     = {'Siena', 'Mendeley'}

# ── Seleção de pacientes ──────────────────────────────────────────────────────
N_SELECT_PER_DS  = {'CHBMIT': 7, 'Siena': 7, 'SeizeIT2': 7, 'Mendeley': 3}
# PN03: excluído manualmente (sem contextos suficientes após validação)
# PN08: arquivo EDF corrompido — excluído
# PN010: pasta no disco com nome errado, mesmo paciente que PN10 — excluído via nome
EXCLUDE_PATIENTS = {'Siena': ['PN03', 'PN08', 'PN010']}

# ── URLs / credenciais por fonte ──────────────────────────────────────────────
PHYSIONET      = 'https://physionet.org/files'
URL_CHBMIT     = f'{PHYSIONET}/chbmit/1.0.0'
URL_SIENA      = f'{PHYSIONET}/siena-scalp-eeg/1.0.0'
PHYSIONET_USER = ''   # deixar vazio — datasets são open access
PHYSIONET_PASS = ''

# SeizeIT2 — OpenNeuro S3 anônimo (bucket openneuro.org, dataset ds005873)
SEIZEIT_BUCKET  = 'openneuro.org'
SEIZEIT_DATASET = 'ds005873'
URL_SEIZEIT     = f's3://{SEIZEIT_BUCKET}/{SEIZEIT_DATASET}'

# Mendeley Data — API pública (cloudscraper contorna Cloudflare)
MEND_DATASET_ID = '5pc2j46cbc'
MEND_VERSION    = 1
MEND_EDF_FOLDER = '3bc6cc31-0156-490b-8e39-4b740598b289'
MEND_DOC_FOLDER = '465c0896-7d08-4fbe-8ee3-378beca656d5'
MEND_HEADERS    = {'Accept': 'application/vnd.mendeley-public-dataset.1+json'}
MEND_API        = f'https://data.mendeley.com/public-api/datasets/{MEND_DATASET_ID}'
URL_MENDELEY    = f'{MEND_API}/files?folder_id={MEND_EDF_FOLDER}&version={MEND_VERSION}'

print('Parâmetros carregados.')


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Parâmetros carregados.


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Teste de conexão com cada fonte

In [3]:
def _http_get(url, timeout=15, auth=None, headers=None, max_bytes=None):
    """GET simples; retorna (status, texto) ou (None, erro).
    max_bytes=None lê a resposta inteira (necessário para listagens de
    diretório, que tem cabeçalho HTML antes da lista de pastas/arquivos —
    um limite pequeno cortava o conteúdo antes de chegar nos links reais).
    """
    try:
        req = urllib.request.Request(url, headers=headers or {})
        if auth and auth[0]:
            import base64
            creds = base64.b64encode(f'{auth[0]}:{auth[1]}'.encode()).decode()
            req.add_header('Authorization', f'Basic {creds}')
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = r.read() if max_bytes is None else r.read(max_bytes)
            return r.status, body.decode('utf-8', errors='ignore')
    except urllib.error.HTTPError as e:
        return e.code, str(e)
    except Exception as e:
        return None, str(e)

def _s3_client():
    """Cliente S3 anônimo (funciona para OpenNeuro e PhysioNet S3)."""
    return boto3.client('s3', region_name='us-east-1',
                        config=Config(signature_version=UNSIGNED))

def _test_s3(bucket, prefix, timeout=15):
    """Testa acesso S3 anônimo listando 1 objeto sob o prefixo."""
    try:
        s3 = boto3.client('s3', region_name='us-east-1',
                          config=Config(signature_version=UNSIGNED,
                                        connect_timeout=timeout, read_timeout=timeout))
        r = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=1)
        return True, f'{r.get("KeyCount", 0)} objeto(s) encontrado(s)'
    except Exception as e:
        return False, str(e)

def _test_mendeley(timeout=15):
    """Testa API pública do Mendeley Data via cloudscraper."""
    try:
        scraper = cloudscraper.create_scraper()
        r = scraper.get(URL_MENDELEY, headers=MEND_HEADERS, timeout=timeout)
        return r.status_code < 400, f'HTTP {r.status_code}'
    except Exception as e:
        return False, str(e)

print('TESTE DE CONEXÃO')
print('='*62)

results = {}

# PhysioNet — HTTP simples
for name, url in [('CHB-MIT  (PhysioNet HTTP)', URL_CHBMIT),
                  ('Siena    (PhysioNet HTTP)', URL_SIENA)]:
    status, _ = _http_get(url, timeout=10,
                          auth=(PHYSIONET_USER, PHYSIONET_PASS) if PHYSIONET_USER else None)
    ok = status is not None and status < 400
    results[name] = ok
    print(f'  {"✓" if ok else "✗"} {name}  →  HTTP {status}')

# SeizeIT2 — OpenNeuro S3 anônimo (boto3, não HTTP)
ok_sz, msg_sz = _test_s3(SEIZEIT_BUCKET, f'{SEIZEIT_DATASET}/')
results['SeizeIT2 (OpenNeuro S3)'] = ok_sz
print(f'  {"✓" if ok_sz else "✗"} SeizeIT2 (OpenNeuro S3)   →  {msg_sz}')

# Mendeley — API pública via cloudscraper (não urllib puro)
ok_me, msg_me = _test_mendeley()
results['Mendeley (API cloudscraper)'] = ok_me
print(f'  {"✓" if ok_me else "✗"} Mendeley (API cloudscraper) →  {msg_me}')

print()
if all(results.values()):
    print('Todas as fontes acessíveis — prosseguindo.')
else:
    falhas = [k for k, v in results.items() if not v]
    print(f'⚠ Fontes com falha: {falhas}')
    print('  Verifique conexão/VPN/firewall antes de continuar.')


TESTE DE CONEXÃO
  ✓ CHB-MIT  (PhysioNet HTTP)  →  HTTP 200
  ✓ Siena    (PhysioNet HTTP)  →  HTTP 200
  ✓ SeizeIT2 (OpenNeuro S3)   →  1 objeto(s) encontrado(s)
  ✓ Mendeley (API cloudscraper) →  HTTP 200

Todas as fontes acessíveis — prosseguindo.


## 2. Funções de acesso remoto (listagem, download, duração EDF)

In [4]:
def _parse_physionet_listing(html):
    """Extrai nomes de arquivos/pastas do HTML de listagem do PhysioNet.
    Pastas aparecem como href="chb01/" (com barra final) — o regex antigo
    excluía '/' do valor e nunca casava com pastas, só arquivos soltos.
    """
    raw = re.findall(r'href="([^"?#]+)"', html)
    return [r for r in raw if not r.startswith(('http://', 'https://', '../', '?'))]

def _download_file(url, local_path, auth=None, timeout=120):
    """Baixa arquivo HTTP simples com checkpoint tmp→rename."""
    local_path = Path(local_path)
    if local_path.exists():
        return True, 'já existe'
    local_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = local_path.with_suffix(local_path.suffix + '.tmp')
    if tmp.exists(): tmp.unlink()
    try:
        req = urllib.request.Request(url)
        if auth and auth[0]:
            import base64
            creds = base64.b64encode(f'{auth[0]}:{auth[1]}'.encode()).decode()
            req.add_header('Authorization', f'Basic {creds}')
        with urllib.request.urlopen(req, timeout=timeout) as r:
            chunk = 1 << 20
            with open(tmp, 'wb') as f:
                while True:
                    data = r.read(chunk)
                    if not data: break
                    f.write(data)
        tmp.rename(local_path)
        return True, f'{local_path.stat().st_size / 1e6:.1f} MB'
    except Exception as e:
        if tmp.exists(): tmp.unlink()
        return False, str(e)

def _download_s3(bucket, key, local_path):
    """Baixa arquivo de S3 anônimo (OpenNeuro)."""
    local_path = Path(local_path)
    if local_path.exists():
        return True, 'já existe'
    local_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        _s3_client().download_file(bucket, key, str(local_path))
        return True, f'{local_path.stat().st_size / 1e6:.1f} MB'
    except Exception as e:
        return False, str(e)

def _download_mendeley(download_url, local_path, timeout=120):
    """Baixa arquivo do Mendeley via cloudscraper."""
    local_path = Path(local_path)
    if local_path.exists():
        return True, 'já existe'
    local_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        scraper = cloudscraper.create_scraper()
        r = scraper.get(download_url, timeout=timeout)
        r.raise_for_status()
        with open(local_path, 'wb') as f:
            f.write(r.content)
        return True, f'{local_path.stat().st_size / 1e6:.1f} MB'
    except Exception as e:
        return False, str(e)

# ── Cache do índice de arquivos Mendeley (filename → download_url) ───────────
_MEND_URL_CACHE = {}

def _mendeley_file_index():
    """Consulta API Mendeley e retorna {filename: download_url}. Cacheado."""
    if _MEND_URL_CACHE:
        return _MEND_URL_CACHE
    try:
        scraper = cloudscraper.create_scraper()
        r = scraper.get(URL_MENDELEY, headers=MEND_HEADERS, timeout=30)
        r.raise_for_status()
        for fobj in r.json():
            _MEND_URL_CACHE[fobj['filename']] = fobj['content_details']['download_url']
    except Exception as e:
        print(f'  ⚠ Mendeley API: {e}')
    return _MEND_URL_CACHE

def _edf_duration_header_bytes(hdr):
    """Extrai duração (s) dos 512 primeiros bytes de um header EDF."""
    if len(hdr) < 252: return None
    try:
        n_records  = int(hdr[236:244].decode('latin-1').strip())
        record_dur = float(hdr[244:252].decode('latin-1').strip())
        return float(n_records * record_dur)
    except Exception:
        return None

def _edf_duration_local(path):
    """Lê duração do EDF de arquivo local (só header)."""
    try:
        with open(path, 'rb') as f:
            hdr = f.read(512)
        return _edf_duration_header_bytes(hdr)
    except Exception:
        return None

def _edf_duration_http(url, auth=None):
    """Lê duração via HTTP Range request (PhysioNet)."""
    try:
        req = urllib.request.Request(url)
        req.add_header('Range', 'bytes=0-511')
        if auth and auth[0]:
            import base64
            creds = base64.b64encode(f'{auth[0]}:{auth[1]}'.encode()).decode()
            req.add_header('Authorization', f'Basic {creds}')
        with urllib.request.urlopen(req, timeout=15) as r:
            hdr = r.read(512)
        return _edf_duration_header_bytes(hdr)
    except Exception:
        return None

def _edf_duration_s3(bucket, key):
    """Lê duração via S3 GetObject Range (OpenNeuro)."""
    try:
        r   = _s3_client().get_object(Bucket=bucket, Key=key, Range='bytes=0-511')
        hdr = r['Body'].read(512)
        return _edf_duration_header_bytes(hdr)
    except Exception:
        return None

def edf_duration(path, ref=None, auth=None):
    """Retorna duração em segundos. `ref` pode ser:
      - URL HTTP normal (PhysioNet)
      - ('s3', bucket, key) para OpenNeuro
      - URL Mendeley (download completo é a única opção — sem Range confiável)
    Tenta local primeiro, sempre.
    """
    if path and Path(path).exists():
        d = _edf_duration_local(path)
        if d: return d
    if ref is None:
        return None
    if isinstance(ref, tuple) and ref[0] == 's3':
        return _edf_duration_s3(ref[1], ref[2])
    if isinstance(ref, str) and ref.startswith('http'):
        return _edf_duration_http(ref, auth=auth)
    return None

print('Funções de acesso remoto definidas.')
print('  PhysioNet  → urllib + Range HTTP')
print('  OpenNeuro  → boto3 S3 anônimo + Range GetObject')
print('  Mendeley   → cloudscraper + API pública')


Funções de acesso remoto definidas.
  PhysioNet  → urllib + Range HTTP
  OpenNeuro  → boto3 S3 anônimo + Range GetObject
  Mendeley   → cloudscraper + API pública


## 3. Ground truth de anotações (Siena e Mendeley)

In [5]:
# Siena: onsets em segundos relativos ao início do arquivo
# Calculados como: (seizure_start_time - registration_start_time) % 86400
# Fonte: arquivos Seizures-list-PNXX.txt de cada paciente
SIENA_GT = {
    'PN00': {
        'pn00-1.edf': [(1143, 1213)],
        'pn00-2.edf': [(1220, 1274)],
        # crise 3: txt indica end=19:29:29 (63min) — provavelmente typo (19→18)
        # usamos onset correto e offset conservador (+120s)
        'pn00-3.edf': [(765, 885)],
        'pn00-4.edf': [(1006, 1080)],
        'pn00-5.edf': [(904, 971)],
    },
    'PN01': {
        # PN01 tem arquivo único (PN01.edf) com 2 crises
        'pn01.edf':   [(10218, 10272), (46353, 46427)],
    },
    'PN05': {
        'pn05-2.edf': [(7163, 7198)],
        'pn05-3.edf': [(6836, 6866)],
        'pn05-4.edf': [(3608, 3647)],
    },
    'PN06': {
        # ATENÇÃO: txt usa 'PNO6' (letra O) mas servidor PhysioNet usa 'pn06' (zero)
        # normalize_fn() converte automaticamente; GT usa nomes corretos (zero)
        'pn06-1.edf': [(5583, 5647)],
        'pn06-2.edf': [(8860, 8929)],
        'pn06-3.edf': [(6275, 6317)],
        'pn06-4.edf': [(5939, 6002)],
        'pn06-5.edf': [(4783, 4827)],
    },
    'PN07': {
        # Gravação longa (23:18→08:01, ~8.7h), crise às 05:25 → 367min no arquivo
        'pn07-1.edf': [(22059, 22121)],
    },
    'PN09': {
        'pn09-1.edf': [(7249, 7329)],
        'pn09-2.edf': [(7127, 7186)],
        'pn09-3.edf': [(7221, 7285)],
    },
    'PN10': {
        'pn10-1.edf':       [(7545, 7614)],
        'pn10-2.edf':       [(7798, 7849)],
        'pn10-3.edf':       [(7835, 7904)],
        'pn10-4.5.6.edf':   [(2309, 2314), (6544, 6563), (11225, 11282)],
        'pn10-7.8.9.edf':   [(2748, 2796), (5459, 5477), (12923, 12938)],
        'pn10-10.edf':      [(7977, 7991)],
    },
    'PN13': {
        'pn13-1.edf': [(7062, 7110)],
        'pn13-2.edf': [(7249, 7314)],
        'pn13-3.edf': [(7553, 7704)],
    },
    'PN14': {
        'pn14-1.edf': [(7262, 7289)],
        'pn14-2.edf': [(7479, 7491)],
        'pn14-3.edf': [(17540, 17581)],
        'pn14-4.edf': [(5463, 5546)],
    },
    'PN17': {
        'pn17-1.edf': [(8420, 8490)],
        'pn17-2.edf': [(7731, 7814)],
    },
}

# Mendeley: onsets em segundos (hardcoded — 6 pacientes conhecidos)
MENDELEY_GT = {
    'p10': {'p10_record1.edf': [(7199, 7644)],
            'p10_record2.edf': [(7200, 7505)]},
    'p13': {'p13_record1.edf': [(7196, 7248)],
            'p13_record2.edf': [(3241, 3266), (7196, 7212)],
            'p13_record3.edf': [(7240, 7258)],
            'p13_record4.edf': [(1450, 1480), (6651, 6675)]},
    'p15': {'p15_record1.edf': [(7158, 7208)],
            'p15_record2.edf': [(7147, 7194)],
            'p15_record3.edf': [(7132, 7145)],
            'p15_record4.edf': [(2431, 2487), (7234, 7254)]},
}

print(f'Ground truth carregado (Siena e Mendeley).')
print(f'  Siena:    {len(SIENA_GT)} pacientes')
print(f'  Mendeley: {len(MENDELEY_GT)} pacientes')


Ground truth carregado (Siena e Mendeley).
  Siena:    10 pacientes
  Mendeley: 3 pacientes


## 4. Funções auxiliares

In [6]:
def _hms_to_s(hms_str):
    """HH:MM:SS ou HH.MM.SS → segundos desde meia-noite (int)."""
    parts = re.split(r'[.:]', hms_str.strip())
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

def m(s):
    return round(s / 60, 1) if s is not None else None

def fmt_hms(s):
    if s is None: return '--'
    s = int(round(float(s)))
    return f'{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}'

def normalize_fn(f):
    """Normaliza nome de arquivo: lowercase, hífens duplos removidos.
    Para Siena: substitui letra 'o' por '0' em nomes de paciente (ex: pno6 → pn06).
    O PhysioNet usa zero numérico nos nomes reais de arquivo.
    """
    f = re.sub(r'-+', '-', f.lower())
    # Corrige confusão letra-o / zero em nomes Siena (ex: pno6 → pn06)
    f = re.sub(r'^(pn)o(\d)', r'\g<1>0\2', f)
    return f

def siena_match(fn, gt):
    """Busca fn no ground-truth gt, tolerando confusão o↔0 em nomes Siena."""
    k = normalize_fn(fn)   # já normaliza pno6→pn06
    if k in gt: return gt[k]
    # Tenta variantes o↔0 como fallback extra
    parts = k.split('-', 1)
    if len(parts) == 2:
        p, s = parts
        for v in [p.replace('0','o'), p.replace('o','0')]:
            if v+'-'+s in gt: return gt[v+'-'+s]
    return []

def parse_chbmit_summary_text(text):
    """Parseia texto do summary CHB-MIT. Tempos já em segundos (int)."""
    sz_map, free = {}, []
    for block in text.split('File Name:')[1:]:
        lines = block.strip().split('\n')
        fname = lines[0].strip()
        n_sz = 0; onsets = []; offsets = []
        for ln in lines:
            if 'Number of Seizures' in ln:
                try: n_sz = int(re.search(r'\d+', ln.split(':',1)[1]).group())
                except: pass
            elif 'Seizure' in ln and 'Start Time' in ln:
                mx = re.search(r'(\d+)', ln.split(':',1)[1])
                if mx: onsets.append(int(mx.group()))
            elif 'Seizure' in ln and 'End Time' in ln:
                mx = re.search(r'(\d+)', ln.split(':',1)[1])
                if mx: offsets.append(int(mx.group()))
        if n_sz > 0:
            sz_map[fname] = [(o, e) for o, e in zip(onsets, offsets) if e > o]
        else:
            free.append(fname)
    return sz_map, free

def parse_siena_annotation_text(text):
    """Parseia Seizures-list-PNXX.txt. Converte HH:MM:SS → segundos.
    Retorna {edf_name_lower: [(onset_s, offset_s), ...]}
    """
    def _parse_time(t_str):
        t_str = t_str.strip()
        t_str = re.split(r'[;(]|\bopure\b|\bor\b', t_str, flags=re.I)[0].strip()
        t_str = re.sub(r'(\d)\s+(\d)', r'\1\2', t_str)
        t_str = re.sub(r'[.\-]', ':', t_str)
        mm = re.match(r'(\d{1,2}):(\d{2}):(\d{2})', t_str)
        return int(mm.group(1))*3600 + int(mm.group(2))*60 + int(mm.group(3)) if mm else None

    result = {}
    for block in re.split(r'Seizure\s+n[°.]?\s*\d+', text, flags=re.IGNORECASE)[1:]:
        fname = rec_s = sz_on = sz_off = None
        for ln in block.split('\n'):
            ln = ln.strip()
            mm = re.search(r'(?i)file\s+name\s*:\s*(\S+\.edf)', ln)
            if mm: fname = normalize_fn(mm.group(1)); continue  # normalize_fn: pno6→pn06
            mm = re.search(r'(?i)registration\s+start\s+time\s*:\s*(.+)', ln)
            if mm: rec_s = _parse_time(mm.group(1)); continue
            mm = re.search(r'(?i)seizure\s+start\s+time\s*:\s*(.+)', ln)
            if mm: sz_on = _parse_time(mm.group(1)); continue
            mm = re.search(r'(?i)seizure\s+end\s+time\s*:\s*(.+)', ln)
            if mm: sz_off = _parse_time(mm.group(1)); continue
        if fname and rec_s is not None and sz_on is not None and sz_off is not None:
            onset_s  = (sz_on  - rec_s + 86400) % 86400
            offset_s = (sz_off - rec_s + 86400) % 86400
            if offset_s < onset_s: offset_s += 86400
            if 0 < offset_s - onset_s < 600:
                result.setdefault(fname, []).append((int(onset_s), int(offset_s)))
    return result

print('Funções auxiliares definidas.')
print('  Todos os tempos internos em segundos (int/float).')
print('  HH:MM:SS convertido apenas em _hms_to_s() e parse_siena_annotation_text().')


Funções auxiliares definidas.
  Todos os tempos internos em segundos (int/float).
  HH:MM:SS convertido apenas em _hms_to_s() e parse_siena_annotation_text().


## 5. Descoberta remota de pacientes + download de anotações

In [7]:
AUTH = (PHYSIONET_USER, PHYSIONET_PASS) if PHYSIONET_USER else None

def _physionet_list_dirs(base_url, prefix, auth=None):
    """Lista subpastas de uma página PhysioNet filtrando por prefixo."""
    status, html = _http_get(base_url, auth=auth)
    if not status or status >= 400:
        print(f'  ✗ Falha ao listar {base_url}: HTTP {status}')
        return []
    entries = _parse_physionet_listing(html)
    return sorted(e.rstrip('/') for e in entries
                  if e.startswith(prefix) and not e.endswith(('.txt','.csv','.json')))

def discover_all_patients(verbose=True):
    """Descobre pacientes consultando as fontes remotas.
    Cada fonte usa o protocolo correto: HTTP (PhysioNet), S3 (OpenNeuro),
    API+cloudscraper (Mendeley). Baixa anotações para cache local.
    """
    found = {'CHBMIT': [], 'Siena': [], 'SeizeIT2': [], 'Mendeley': []}

    # ── CHB-MIT (PhysioNet HTTP) ──────────────────────────────────────────────
    print('CHB-MIT: listando pacientes no PhysioNet...')
    for pat in _physionet_list_dirs(URL_CHBMIT, 'chb', auth=AUTH):
        local_summary = CHBMIT_DIR / pat / f'{pat}-summary.txt'
        if not local_summary.exists():
            url = f'{URL_CHBMIT}/{pat}/{pat}-summary.txt'
            ok, msg = _download_file(url, local_summary, auth=AUTH)
            if not ok:
                print(f'  ✗ {pat}: falha summary ({msg})'); continue
        found['CHBMIT'].append(pat)
    print(f'  ✓ {len(found["CHBMIT"])} pacientes  (summary.txt em cache)')

    # ── Siena (PhysioNet HTTP) ────────────────────────────────────────────────
    print('Siena: listando pacientes no PhysioNet...')
    for pat in _physionet_list_dirs(URL_SIENA, 'PN', auth=AUTH):
        local_ann = SIENA_DIR / pat / f'Seizures-list-{pat}.txt'
        if not local_ann.exists():
            url = f'{URL_SIENA}/{pat}/Seizures-list-{pat}.txt'
            ok, msg = _download_file(url, local_ann, auth=AUTH)
            if not ok:
                _, html = _http_get(f'{URL_SIENA}/{pat}/', auth=AUTH)
                txts = [e for e in _parse_physionet_listing(html or '')
                        if e.lower().endswith('.txt')]
                if txts:
                    ok2, _ = _download_file(f'{URL_SIENA}/{pat}/{txts[0]}',
                                            SIENA_DIR / pat / txts[0], auth=AUTH)
                    if ok2: local_ann = SIENA_DIR / pat / txts[0]
                    else:
                        print(f'  ✗ {pat}: sem anotação'); continue
                else:
                    print(f'  ✗ {pat}: sem anotação'); continue
        found['Siena'].append(pat)
    print(f'  ✓ {len(found["Siena"])} pacientes  (annotation .txt em cache)')

    # ── SeizeIT2 (OpenNeuro S3 anônimo) ───────────────────────────────────────
    print('SeizeIT2: listando pacientes no OpenNeuro S3...')
    s3 = _s3_client()
    resp = s3.list_objects_v2(Bucket=SEIZEIT_BUCKET,
                              Prefix=f'{SEIZEIT_DATASET}/', Delimiter='/')
    sz2_pats = sorted(
        p['Prefix'].rstrip('/').split('/')[-1]
        for p in resp.get('CommonPrefixes', [])
        if re.match(r'sub-\d+', p['Prefix'].rstrip('/').split('/')[-1])
    )
    paginator = s3.get_paginator('list_objects_v2')
    for pat in sz2_pats:
        prefix  = f'{SEIZEIT_DATASET}/{pat}/{SEIZEIT_SESSION}/eeg/'
        eeg_dir = SEIZEIT_DIR / pat / SEIZEIT_SESSION / 'eeg'
        eeg_dir.mkdir(parents=True, exist_ok=True)
        keys = []
        for page in paginator.paginate(Bucket=SEIZEIT_BUCKET, Prefix=prefix):
            keys.extend(o['Key'] for o in page.get('Contents', []))
        tsv_keys = [k for k in keys if k.endswith('_events.tsv')]
        if not tsv_keys: continue
        downloaded = 0
        for key in tsv_keys:
            fn = key.split('/')[-1]
            local = eeg_dir / fn
            if not local.exists():
                ok, msg = _download_s3(SEIZEIT_BUCKET, key, local)
                if not ok:
                    print(f'    ✗ {fn}: {msg}'); continue
            downloaded += 1
        if downloaded > 0:
            found['SeizeIT2'].append(pat)
    print(f'  ✓ {len(found["SeizeIT2"])} pacientes  (*_events.tsv em cache)')

    # ── Mendeley (API pública + cloudscraper) ─────────────────────────────────
    print('Mendeley: consultando API pública (cloudscraper)...')
    idx = _mendeley_file_index()   # aquece o cache e valida acesso
    found['Mendeley'] = sorted(MENDELEY_GT.keys())
    print(f'  ✓ {len(found["Mendeley"])} pacientes  '
          f'({len(idx)} arquivos indexados via API)')

    total = sum(len(v) for v in found.values())
    print(f'\nTotal descoberto: {total} pacientes')
    return found

print('discover_all_patients() definida.')


discover_all_patients() definida.


## 6. Carregamento de metadados dos EDFs (com duração real)

In [8]:
def _edf_ref(dataset, local_path):
    """Retorna a referência remota de um EDF, no formato que `edf_duration`
    e o downloader esperam para cada fonte:
      - CHBMIT/Siena → URL HTTP (PhysioNet)
      - SeizeIT2     → ('s3', bucket, key)
      - Mendeley     → URL de download (Mendeley API) ou None
    """
    p = Path(local_path)
    if dataset == 'CHBMIT':
        return f'{URL_CHBMIT}/{p.parent.name}/{p.name}'
    elif dataset == 'Siena':
        # PhysioNet Siena serve arquivos em MAIÚSCULAS: PN06/PN06-1.edf
        # PhysioNet Siena: nome maiúsculo mas extensão minúscula (ex: PN06-1.edf)
        return f'{URL_SIENA}/{p.parent.name}/{p.stem.upper()}.edf'
    elif dataset == 'SeizeIT2':
        pat = p.parent.parent.parent.name
        key = f'{SEIZEIT_DATASET}/{pat}/{SEIZEIT_SESSION}/eeg/{p.name}'
        return ('s3', SEIZEIT_BUCKET, key)
    elif dataset == 'Mendeley':
        return _mendeley_file_index().get(p.name)
    return None

def _download_edf_any(dataset, ref, local_path):
    """Baixa um EDF usando o método correto para cada fonte."""
    if ref is None:
        return False, 'referência remota desconhecida'
    if isinstance(ref, tuple) and ref[0] == 's3':
        return _download_s3(ref[1], ref[2], local_path)
    if dataset == 'Mendeley':
        return _download_mendeley(ref, local_path)
    return _download_file(ref, local_path, auth=AUTH)

def load_patient_edfs(dataset, pat):
    """Constrói lista de EDFs com metadados completos.
    dur_s é lido do header EDF (local se disponível, senão via Range remoto).
    Todos os tempos em segundos (int/float).
    """
    edf_list = []

    if dataset == 'CHBMIT':
        summary_path = CHBMIT_DIR / pat / f'{pat}-summary.txt'
        if not summary_path.exists(): return []
        text = summary_path.read_text(errors='ignore')
        sz_map, free_list = parse_chbmit_summary_text(text)
        all_files = list(sz_map.items()) + [(fn, []) for fn in free_list]
        for fn, iv in all_files:
            local = CHBMIT_DIR / pat / fn
            ref   = _edf_ref(dataset, local)
            dur   = edf_duration(local, ref=ref, auth=AUTH)
            edf_list.append({'name': fn, 'path': local, 'ref': ref,
                             'dur_s': dur, 'seizures': iv, 'is_pure': len(iv) == 0})

    elif dataset == 'Siena':
        pat_dir  = SIENA_DIR / pat
        gt_s     = SIENA_GT.get(pat, {})
        ann_file = None
        for g in ['Seizures-list-*.txt', 'Seizure*list*.txt', '*.txt']:
            txts = sorted(pat_dir.glob(g)) if pat_dir.exists() else []
            if txts: ann_file = txts[0]; break
        ann_map = parse_siena_annotation_text(
            ann_file.read_text(errors='ignore') if ann_file else '')
        # Normaliza nomes do ann_map (anotação pode ter 'pno6') e do gt_s
        # para garantir que a URL construída por _edf_ref use '0' (zero)
        known = set(normalize_fn(k) for k in ann_map.keys()) | \
                set(normalize_fn(k) for k in gt_s.keys())
        # Reconstrói ann_map com chaves normalizadas
        ann_map = {normalize_fn(k): v for k, v in ann_map.items()}
        for fn_lower in sorted(known):
            local = pat_dir / fn_lower
            if not local.exists():
                for cand in [pat_dir / fn_lower.upper(),
                              pat_dir / (fn_lower[0].upper() + fn_lower[1:])]:
                    if cand.exists(): local = cand; break
            ref = _edf_ref(dataset, pat_dir / fn_lower)
            dur = edf_duration(local, ref=ref, auth=AUTH)
            iv  = siena_match(fn_lower, gt_s) or ann_map.get(fn_lower, [])
            edf_list.append({'name': fn_lower, 'path': local, 'ref': ref,
                             'dur_s': dur, 'seizures': iv, 'is_pure': len(iv) == 0})

    elif dataset == 'SeizeIT2':
        eeg_dir = SEIZEIT_DIR / pat / SEIZEIT_SESSION / 'eeg'
        if not eeg_dir.exists(): return []
        for tsv in sorted(eeg_dir.glob('*_events.tsv')):
            edf_name = tsv.name.replace('_events.tsv', '_eeg.edf')
            local    = eeg_dir / edf_name
            ref      = _edf_ref(dataset, local)
            dur      = edf_duration(local, ref=ref)
            iv = []
            try:
                df = pd.read_csv(tsv, sep='\t')
                if 'eventType' in df.columns:
                    et = df['eventType'].astype(str).str.lower().str.strip()
                    for _, row in df[et.str.startswith('sz')].iterrows():
                        try:
                            o = float(row['onset']); d = float(row['duration'])
                            if d > 0: iv.append((float(o), float(o + d)))
                        except: pass
            except: pass
            edf_list.append({'name': edf_name, 'path': local, 'ref': ref,
                             'dur_s': dur, 'seizures': iv, 'is_pure': len(iv) == 0})

    elif dataset == 'Mendeley':
        edf_dir = MENDELEY_DIR / 'Raw_EDF_Files'
        if not edf_dir.exists(): edf_dir = MENDELEY_DIR
        for fn, iv in MENDELEY_GT.get(pat, {}).items():
            local = edf_dir / fn
            ref   = _edf_ref(dataset, local)
            dur   = edf_duration(local, ref=ref)
            edf_list.append({'name': fn, 'path': local, 'ref': ref,
                             'dur_s': dur, 'seizures': list(iv), 'is_pure': len(iv) == 0})
    return edf_list

print('load_patient_edfs() definida.')
print('  dur_s sempre em segundos reais (header EDF local ou Range remoto).')
print('  ref carrega a referência remota correta por fonte (URL, S3 ou Mendeley).')


load_patient_edfs() definida.
  dur_s sempre em segundos reais (header EDF local ou Range remoto).
  ref carrega a referência remota correta por fonte (URL, S3 ou Mendeley).


## 7. Validadores de contexto

In [9]:
def validate_dedicated(pat, edf_list):
    """CHB-MIT: interictal de arquivo puro separado. dur_s sempre real."""
    pure_edfs = sorted([e for e in edf_list if e['is_pure']], key=lambda x: x['name'])
    sz_edfs   = sorted([e for e in edf_list if not e['is_pure'] and e['seizures']],
                       key=lambda x: x['name'])
    pool = [{'name': pe['name'], 'path': pe['path'], 'ref': pe.get('ref'),
             'dur_s': pe['dur_s'], 'used_s': 0.0} for pe in pure_edfs]
    pool_idx = 0
    contexts = []

    for edf in sz_edfs:
        seiz = [(float(on), float(off)) for on, off in edf['seizures']]
        for i, (on, off) in enumerate(seiz):
            approved = on >= MAX_PRE_S
            motivo = None
            if not approved:
                motivo = f'onset={m(on)}min < MAX_PRE={m(MAX_PRE_S)}min'
            else:
                pre_ini = on - MAX_PRE_S
                for j, (on_o, off_o) in enumerate(seiz):
                    if j == i: continue
                    if pre_ini < off_o and on > on_o:
                        approved = False; motivo = f'pre-ictal contem crise #{j+1}'; break
                    if off_o < on and pre_ini < off_o + GUARD_S and on > off_o:
                        approved = False; motivo = f'pre-ictal colide pos-ictal crise #{j+1}'; break

            prev_limit   = max((off_o + GUARD_S for _, off_o in seiz if off_o < on), default=0.0)
            pre_max_disp = max(0.0, on - prev_limit)

            if not approved:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                    'dur_edf': edf['dur_s'], 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': False, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': None, 'pre_fim_s': None,
                    'inter_arquivo': None, 'inter_path': None, 'inter_ref': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': '--', 'valid': False, 'motivo': motivo,
                }); continue

            pi = pool_idx; found = False
            while pi < len(pool):
                p = pool[pi]
                dur_p = p['dur_s']
                if dur_p is None:
                    pi += 1; continue
                restante = dur_p - p['used_s']
                if restante >= MIN_INTER_S:
                    ini = p['used_s']; fim = dur_p
                    p['used_s'] = fim; pool_idx = pi
                    contexts.append({
                        'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                        'dur_edf': edf['dur_s'], 'sz_idx': i+1,
                        'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                        'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                        'pre_inicio_s': on - MAX_PRE_S, 'pre_fim_s': on,
                        'inter_arquivo': p['name'], 'inter_path': p['path'],
                        'inter_ref': p.get('ref'),
                        'inter_inicio_s': ini, 'inter_fim_s': fim,
                        'inter_fonte': 'dedicado', 'valid': True,
                        'motivo': f'inter dedicado {p["name"]} [{m(ini)}-{m(fim)}min]',
                    }); found = True; break
                pi += 1; pool_idx = pi
            if not found:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                    'dur_edf': edf['dur_s'], 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': on - MAX_PRE_S, 'pre_fim_s': on,
                    'inter_arquivo': None, 'inter_path': None, 'inter_ref': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': 'esgotado', 'valid': False,
                    'motivo': 'pool de interictal puro esgotado',
                })
    return contexts


def validate_intra(pat, edf_list):
    """Siena / Mendeley / SeizeIT2: interictal intra-arquivo."""
    MIN_ONSET = MAX_PRE_S + GUARD_S
    contexts  = []

    for edf in edf_list:
        if edf['is_pure'] or not edf['seizures']: continue
        dur  = edf['dur_s']
        seiz = [(float(on), float(off)) for on, off in edf['seizures']]

        forbidden = [(max(0.0, on - MAX_PRE_S - GUARD_S), off + GUARD_S) for on, off in seiz]
        forbidden.sort()
        locked = []

        def free_space(start, end):
            blocks = []; cur = start
            for (a, b) in sorted(forbidden + locked):
                if b <= cur: continue
                if a >= end: break
                if a > cur: blocks.append((cur, a))
                cur = max(cur, b)
            if cur < end: blocks.append((cur, end))
            return blocks

        for i, (on, off) in enumerate(seiz):
            zona_ini     = max(0.0, on - MAX_PRE_S - GUARD_S)
            prev_limit   = max((off_o + GUARD_S for _, off_o in seiz if off_o < on), default=0.0)
            pre_max_disp = max(0.0, on - prev_limit)

            if on < MIN_ONSET:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': False, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': None, 'pre_fim_s': None,
                    'inter_arquivo': None, 'inter_path': None, 'inter_ref': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': '--', 'valid': False,
                    'motivo': f'onset={m(on)}min < {m(MIN_ONSET)}min (MAX_PRE+GUARD)',
                }); continue

            pre_ini    = on - MAX_PRE_S
            pre_colide = any(off_o < on and pre_ini < off_o + GUARD_S and on > off_o
                             for _, off_o in seiz)
            if pre_colide:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': False, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': None, 'pre_fim_s': None,
                    'inter_arquivo': None, 'inter_path': None, 'inter_ref': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': '--', 'valid': False,
                    'motivo': 'pre-ictal colide com pos-ictal de crise anterior',
                }); continue

            r1       = free_space(0.0, zona_ini)
            last_seg = r1[-1] if r1 else None
            last_dur = (last_seg[1] - last_seg[0]) if last_seg else 0.0

            if last_dur >= MIN_INTER_S:
                inter_ini = last_seg[0]; inter_fim = last_seg[1]
                fonte = 'antes (R1-último-segmento)'
                locked.append((inter_ini, inter_fim)); locked.sort()
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': pre_ini, 'pre_fim_s': on,
                    'inter_arquivo': edf['name'], 'inter_path': edf['path'],
                    'inter_ref': edf.get('ref'),
                    'inter_inicio_s': inter_ini, 'inter_fim_s': inter_fim,
                    'inter_fonte': fonte, 'valid': True,
                    'motivo': f'{fonte}: inter=[{m(inter_ini)}-{m(inter_fim)}min] pre=[{m(pre_ini)}-{m(on)}min]',
                })
            else:
                contexts.append({
                    'edf': edf['name'], 'path': edf['path'], 'ref': edf.get('ref'),
                    'dur_edf': dur, 'sz_idx': i+1,
                    'on_s': on, 'off_s': off, 'n_sz_edf': len(seiz),
                    'approved_pre': True, 'pre_max_disp_s': pre_max_disp,
                    'pre_inicio_s': pre_ini, 'pre_fim_s': on,
                    'inter_arquivo': None, 'inter_path': None, 'inter_ref': None,
                    'inter_inicio_s': None, 'inter_fim_s': None,
                    'inter_fonte': 'insuf', 'valid': False,
                    'motivo': f'inter insuf: R1_ultimo_seg={m(last_dur)}min < {m(MIN_INTER_S)}min',
                })
    return contexts

VALIDATORS = {
    'CHBMIT':   validate_dedicated,
    'SeizeIT2': validate_intra,
    'Siena':    validate_intra,
    'Mendeley': validate_intra,
}

print('Validadores definidos (todos os tempos em segundos reais).')


Validadores definidos (todos os tempos em segundos reais).


## 8. Descoberta, validação e seleção de pacientes

In [10]:
# ── Cache de descoberta em disco ──────────────────────────────────────────────
# Evita re-consultar PhysioNet/OpenNeuro/Mendeley e re-baixar anotações toda vez
# que o notebook é reexecutado. Se o cache existir e FORCE_REDISCOVER=False,
# carrega tudo do disco — sem nenhuma chamada de rede nesta célula.
DISCOVERY_CACHE  = ROOT / 'discovery_cache.json'
FORCE_REDISCOVER = False   # mude para True se quiser forçar nova consulta remota

# ── Limpa entradas inválidas do cache (nomes com letra 'o' em vez de '0') ──
# Necessário quando o cache foi gerado antes da correção do normalize_fn.
import re as _re
_BAD_FN = _re.compile(r'pno\d', _re.I)  # detecta pno6, PNO6, etc

def _fix_ctx_ref(c):
    """Corrige ref/inter_ref e path/inter_path que contenham pno→pn0."""
    c = dict(c)
    for k in ('path', 'inter_path'):
        if c.get(k) and _BAD_FN.search(str(c[k])):
            c[k] = _re.sub(r'(?i)pno(\d)', r'pn0\1', str(c[k]))
    for k in ('ref', 'inter_ref'):
        if isinstance(c.get(k), str) and _BAD_FN.search(c[k]):
            c[k] = _re.sub(r'(?i)pno(\d)', r'pn0\1', c[k])
            # Garante maiúsculas no nome do arquivo na URL
            c[k] = _re.sub(r'/(pn\d[^/]*\.edf)', lambda m: '/' + m.group(1).upper().replace('.EDF','.edf'), c[k])
    return c

def _fix_edf_ref(e):
    e = dict(e)
    if e.get('name') and _BAD_FN.search(e['name']):
        e['name'] = _re.sub(r'(?i)pno(\d)', r'pn0\1', e['name'])
    if isinstance(e.get('ref'), str) and _BAD_FN.search(e['ref']):
        e['ref'] = _re.sub(r'(?i)pno(\d)', r'pn0\1', e['ref'])
        e['ref'] = _re.sub(r'/(pn\d[^/]*\.edf)', lambda m: '/' + m.group(1).upper().replace('.EDF','.edf'), e['ref'])
    return e

def _ctx_to_json(c):
    """Serializa um contexto (dict com Path) para JSON."""
    out = dict(c)
    for k in ('path', 'inter_path'):
        if out.get(k) is not None:
            out[k] = str(out[k])
    # 'ref'/'inter_ref' podem ser tuplas ('s3', bucket, key) — json trata ok
    return out

def _ctx_from_json(c):
    """Reconstrói um contexto a partir do JSON (Path de volta)."""
    out = dict(c)
    for k in ('path', 'inter_path'):
        if out.get(k) is not None:
            out[k] = Path(out[k])
    for k in ('ref', 'inter_ref'):
        if isinstance(out.get(k), list):
            out[k] = tuple(out[k])  # json vira lista; ref s3 precisa ser tupla
    return out

def _edf_to_json(e):
    out = dict(e)
    if out.get('path') is not None:
        out['path'] = str(out['path'])
    if isinstance(out.get('ref'), tuple):
        out['ref'] = list(out['ref'])
    return out

def _edf_from_json(e):
    out = dict(e)
    if out.get('path') is not None:
        out['path'] = Path(out['path'])
    if isinstance(out.get('ref'), list):
        out['ref'] = tuple(out['ref'])
    return out


if DISCOVERY_CACHE.exists() and not FORCE_REDISCOVER:
    print(f'Cache de descoberta encontrado em {DISCOVERY_CACHE} — lendo do disco (sem rede)...')
    with open(DISCOVERY_CACHE, encoding='utf-8') as f:
        cache = json.load(f)

    ALL_PATIENTS      = cache['ALL_PATIENTS']
    all_edfs          = {ds: {pat: [_fix_edf_ref(_edf_from_json(e)) for e in edfs]
                               for pat, edfs in pats.items()}
                          for ds, pats in cache['all_edfs'].items()}
    all_contexts_full = {ds: {pat: [_fix_ctx_ref(_ctx_from_json(c)) for c in ctxs]
                               for pat, ctxs in pats.items()}
                          for ds, pats in cache['all_contexts_full'].items()}

    total = sum(len(v) for v in ALL_PATIENTS.values())
    print(f'  {total} pacientes carregados do cache.')
    print(f'  (para re-consultar as fontes remotas, defina FORCE_REDISCOVER=True e rode de novo)')

else:
    print('Descobrindo pacientes nas fontes remotas...')
    ALL_PATIENTS = discover_all_patients(verbose=True)

    print('\nCarregando metadados e validando contextos...')
    all_contexts_full = {}
    all_edfs          = {}

    for dataset, pats in ALL_PATIENTS.items():
        all_contexts_full[dataset] = {}
        all_edfs[dataset]          = {}
        print(f'\n=== {dataset} ({len(pats)} pacientes) ===')
        for pat in pats:
            edf_list = load_patient_edfs(dataset, pat)
            all_edfs[dataset][pat] = edf_list
            if not edf_list:
                print(f'  {pat}: sem metadados'); continue
            ctxs  = VALIDATORS[dataset](pat, edf_list)
            all_contexts_full[dataset][pat] = ctxs
            n_val = sum(1 for c in ctxs if c['valid'])
            tag   = 'OK' if n_val >= MIN_CONTEXTS else 'INSUF'
            print(f'  [{tag}] {pat}: {len(ctxs)} crises | {n_val} ctx válidos')

    # Salva cache para as próximas execuções
    cache = {
        'ALL_PATIENTS': ALL_PATIENTS,
        'all_edfs': {ds: {pat: [_edf_to_json(e) for e in edfs]
                          for pat, edfs in pats.items()}
                     for ds, pats in all_edfs.items()},
        'all_contexts_full': {ds: {pat: [_ctx_to_json(c) for c in ctxs]
                                   for pat, ctxs in pats.items()}
                              for ds, pats in all_contexts_full.items()},
    }
    with open(DISCOVERY_CACHE, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=1, default=str)
    print(f'\nCache salvo em {DISCOVERY_CACHE} — próximas execuções pulam a rede.')

# ── Seleção ───────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('SELEÇÃO AUTOMÁTICA — ranking por contextos válidos')
print('='*60)

all_contexts = {}
SELECTED     = {}

for dataset, pat_ctxs in all_contexts_full.items():
    _excl = EXCLUDE_PATIENTS.get(dataset, [])
    eligible = [
        (pat, ctxs, sum(1 for c in ctxs if c['valid']))
        for pat, ctxs in pat_ctxs.items()
        if sum(1 for c in ctxs if c['valid']) >= MIN_CONTEXTS
        and pat not in _excl
    ]
    if _excl: print(f'  (excluídos: {_excl})')

    def _has_edfs(ctxs):
        return any(c.get('valid') and c.get('path') and Path(c['path']).exists()
                   for c in ctxs)
    eligible.sort(key=lambda x: (0 if _has_edfs(x[1]) else 1, -x[2]))

    limit = (N_SELECT_PER_DS.get(dataset, 7)
             if isinstance(N_SELECT_PER_DS, dict) else N_SELECT_PER_DS)
    if limit: eligible = eligible[:limit]

    all_contexts[dataset] = {pat: ctxs for pat, ctxs, _ in eligible}
    SELECTED[dataset]     = [pat for pat, _, _ in eligible]

    print(f'\n{dataset}: {len(eligible)} selecionados')
    for pat, ctxs, n_v in eligible:
        print(f'  {pat:<14} {n_v:3d} ctx válidos / {len(ctxs):3d} crises')

total_pats = sum(len(v) for v in SELECTED.values())
total_ctx  = sum(sum(1 for c in ctxs if c['valid'])
                 for ds in all_contexts.values() for ctxs in ds.values())
print(f'\nTotal: {total_pats} pacientes | {total_ctx} contextos válidos')


Descobrindo pacientes nas fontes remotas...
CHB-MIT: listando pacientes no PhysioNet...
  ✓ 24 pacientes  (summary.txt em cache)
Siena: listando pacientes no PhysioNet...
  ✓ 14 pacientes  (annotation .txt em cache)
SeizeIT2: listando pacientes no OpenNeuro S3...
  ✓ 125 pacientes  (*_events.tsv em cache)
Mendeley: consultando API pública (cloudscraper)...
  ✓ 3 pacientes  (20 arquivos indexados via API)

Total descoberto: 166 pacientes

Carregando metadados e validando contextos...

=== CHBMIT (24 pacientes) ===
  [OK] chb01: 7 crises | 2 ctx válidos
  [OK] chb02: 3 crises | 2 ctx válidos
  [OK] chb03: 7 crises | 3 ctx válidos
  [OK] chb04: 4 crises | 2 ctx válidos
  [OK] chb05: 5 crises | 3 ctx válidos
  [OK] chb06: 10 crises | 7 ctx válidos
  [OK] chb07: 3 crises | 3 ctx válidos
  [OK] chb08: 5 crises | 5 ctx válidos
  [OK] chb09: 4 crises | 4 ctx válidos
  [OK] chb10: 7 crises | 6 ctx válidos
  [INSUF] chb11: 3 crises | 1 ctx válidos
  [OK] chb12: 40 crises | 2 ctx válidos
  [OK] c

## 9. Verificação de integridade

In [11]:
def check_integrity(dataset, pat, edf_list, contexts):
    problems = []
    sz_by_file = {e['name']: [(float(on), float(off)) for on, off in e['seizures']]
                  for e in edf_list}
    valid_ctxs = [c for c in contexts if c['valid']]

    for c in valid_ctxs:
        if c['inter_inicio_s'] is None: continue
        arq = c['inter_arquivo']
        ii, if_ = c['inter_inicio_s'], c['inter_fim_s']
        for (con, coff) in sz_by_file.get(arq, []):
            if ii < coff and if_ > con:
                problems.append(f'{pat} ctx#{c["sz_idx"]}: interictal contem crise em {arq}')

    for c in valid_ctxs:
        if c['pre_inicio_s'] is None: continue
        pre_i, pre_f = c['pre_inicio_s'], c['pre_fim_s']
        for (con, coff) in sz_by_file.get(c['edf'], []):
            if coff >= c['on_s']: continue
            if pre_i < coff + GUARD_S and pre_f > coff:
                problems.append(f'{pat} ctx#{c["sz_idx"]}: pre-ictal colide pos-ictal anterior')

    inter_by_file = defaultdict(list)
    for c in valid_ctxs:
        if c['inter_inicio_s'] is not None:
            inter_by_file[c['inter_arquivo']].append(
                (c['inter_inicio_s'], c['inter_fim_s'], c['sz_idx']))
    for arq, ivs in inter_by_file.items():
        ivs.sort()
        for a in range(len(ivs)-1):
            if ivs[a][1] > ivs[a+1][0]:
                problems.append(f'{pat}: interictais sobrepostos em {arq}: '
                                 f'ctx#{ivs[a][2]} e ctx#{ivs[a+1][2]}')
    return problems

print('VERIFICAÇÃO DE INTEGRIDADE')
print('='*55)
total_problems = 0
for dataset, pats in SELECTED.items():
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        probs = check_integrity(dataset, pat,
                                all_edfs[dataset][pat],
                                all_contexts[dataset][pat])
        if probs:
            total_problems += len(probs)
            for p in probs: print(f'  PROBLEMA: {p}')

print('TUDO OK — nenhuma violação encontrada.' if total_problems == 0
      else f'{total_problems} problema(s) encontrado(s).')


VERIFICAÇÃO DE INTEGRIDADE
TUDO OK — nenhuma violação encontrada.


## 10. Download de EDFs e salvamento em .npz

In [12]:
def load_and_preprocess(path, tmin_s, tmax_s, apply_car=False):
    if not HAS_MNE or path is None: return None, None, None
    try:
        raw   = mne.io.read_raw_edf(str(path), preload=False, verbose=False)
        sf    = raw.info['sfreq']
        tmin_s = max(0.0, float(tmin_s))
        tmax_s = min((raw.n_times - 1) / sf, float(tmax_s))
        if tmax_s <= tmin_s: raw.close(); return None, None, None

        non_eeg = {'spo2','hr','ekg','ecg','emg','eog','mk','status',
                   'annot','trig','stim','resp','temp'}
        eeg_idx = [i for i, ch in enumerate(raw.ch_names)
                   if ch.upper().startswith('EEG')]
        if not eeg_idx:
            eeg_idx = [i for i, ch in enumerate(raw.ch_names)
                       if not any(kw in ch.lower() for kw in non_eeg)]
        if not eeg_idx: eeg_idx = list(range(len(raw.ch_names)))

        ch_names = [raw.ch_names[i] for i in eeg_idx]
        raw_crop = raw.copy().crop(tmin=tmin_s, tmax=tmax_s)
        raw_crop.pick(eeg_idx)
        raw_crop.load_data(verbose=False)

        clip_v = CLIP_UV * 1e-6
        raw_crop._data = np.clip(raw_crop._data, -clip_v, clip_v)
        raw_crop.filter(l_freq=F_HIGHPASS, h_freq=F_LOWPASS, method='iir', verbose=False)
        raw_crop.notch_filter(freqs=NOTCH_FREQS, verbose=False)
        if abs(sf - TARGET_SFREQ) > 1:
            raw_crop.resample(TARGET_SFREQ, verbose=False)

        data = (raw_crop.get_data() * 1e6).astype(np.float32)
        sfreq_out = raw_crop.info['sfreq']
        raw_crop.close(); raw.close()

        if apply_car and data.shape[0] >= 8:
            data -= data.mean(axis=0, keepdims=True)
        return data, ch_names, sfreq_out
    except Exception as e:
        print(f'    Erro {Path(str(path)).name}: {e}')
        return None, None, None


def npz_valido(path):
    """True se .npz existe e contém todas as chaves esperadas."""
    try:
        d = np.load(path, allow_pickle=True)
        for key in ['inter', 'pre', 'ch_names', 'sfreq', 'meta']:
            _ = d[key]
        return True
    except Exception:
        return False


def save_context_npz(dataset, pat, ctx_id, ctx):
    """Baixa EDFs se necessário (usando o método certo por fonte) e salva .npz."""
    out_path = SIGNAL_DIR / f'{dataset}__{pat}__ctx{ctx_id:03d}.npz'

    if out_path.exists():
        if npz_valido(out_path):
            return True, 'ja existe'
        print(f'    {out_path.name}: corrompido — regenerando')
        try: out_path.unlink()
        except Exception as e:
            return False, f'corrompido mas nao removivel: {e}'

    # Garante EDFs em disco (crise + interictal)
    for p_key, r_key in [('path', 'ref'), ('inter_path', 'inter_ref')]:
        p = ctx.get(p_key)
        r = ctx.get(r_key)
        if p and not Path(str(p)).exists():
            ok, msg = _download_edf_any(dataset, r, p)
            if not ok:
                return False, f'download falhou ({Path(str(p)).name}): {msg}'

    car = dataset in DATASETS_CAR

    inter_ini_cap = max(float(ctx['inter_inicio_s']),
                        float(ctx['inter_fim_s']) - MAX_INTER_SAVE_S)
    inter_data, inter_chs, sfreq = load_and_preprocess(
        ctx['inter_path'], inter_ini_cap, ctx['inter_fim_s'], apply_car=car)
    pre_data, pre_chs, _ = load_and_preprocess(
        ctx['path'], ctx['pre_inicio_s'], ctx['pre_fim_s'], apply_car=car)

    if inter_data is None or pre_data is None:
        return False, 'falha ao carregar EDF'

    common    = [c for c in inter_chs if c in pre_chs] or \
                inter_chs[:min(len(inter_chs), len(pre_chs))]
    inter_idx = [inter_chs.index(c) for c in common if c in inter_chs]
    pre_idx   = [pre_chs.index(c)   for c in common if c in pre_chs]
    if not inter_idx or not pre_idx:
        return False, 'sem canais em comum'

    np.savez_compressed(
        out_path,
        inter    = inter_data[inter_idx],
        pre      = pre_data[pre_idx],
        ch_names = np.array(common),
        sfreq    = np.array(sfreq),
        meta     = np.array(json.dumps({
            'dataset': dataset, 'paciente': pat, 'contexto_id': ctx_id,
            'arquivo_crise': ctx['edf'],
            'onset_s': ctx['on_s'], 'offset_s': ctx['off_s'],
            'inter_arquivo': ctx['inter_arquivo'],
            'inter_inicio_s': ctx['inter_inicio_s'],
            'inter_fim_s': ctx['inter_fim_s'],
            'inter_fonte': ctx['inter_fonte'],
            'pre_inicio_s': ctx['pre_inicio_s'],
            'pre_fim_s': ctx['pre_fim_s'],
            'pre_max_disp_s': ctx['pre_max_disp_s'],
            'guard_s': GUARD_S, 'max_pre_s': MAX_PRE_S,
            'target_sfreq': TARGET_SFREQ,
        }))
    )
    return True, out_path.name


print('Salvando contextos em .npz...\n')
saved = regenerated = skipped = failed = 0

for dataset, pats in SELECTED.items():
    print(f'--- {dataset} ---')
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        ctx_id = 0
        for c in all_contexts[dataset][pat]:
            if not c['valid']: continue
            ctx_id += 1
            ok, msg = save_context_npz(dataset, pat, ctx_id, c)
            if 'ja existe'     in msg: skipped     += 1
            elif 'regenerando' in msg: regenerated += 1
            elif ok:                   saved       += 1; print(f'  {pat} ctx{ctx_id:03d}: {msg}')
            else:                      failed      += 1; print(f'  {pat} ctx{ctx_id:03d}: FALHOU — {msg}')

print(f'\nSalvos: {saved} | Regenerados: {regenerated} | Já existiam: {skipped} | Falhas: {failed}')
print(f'Total .npz em {SIGNAL_DIR}: {len(list(SIGNAL_DIR.glob("*.npz")))}')


Salvando contextos em .npz...

--- CHBMIT ---
--- Siena ---
  PN06 ctx002: Siena__PN06__ctx002.npz
  PN06 ctx003: Siena__PN06__ctx003.npz
  PN06 ctx004: Siena__PN06__ctx004.npz
  PN16 ctx001: Siena__PN16__ctx001.npz
  PN16 ctx002: Siena__PN16__ctx002.npz
--- SeizeIT2 ---
--- Mendeley ---

Salvos: 5 | Regenerados: 0 | Já existiam: 118 | Falhas: 0
Total .npz em data\signals: 125


## 11. Checagem de status por paciente

In [13]:
print(f'{"Dataset":<12} {"Paciente":<14} {"EDFs":<8} {"npz":<10} {"Status"}')
print('─'*60)
ok_pats = warn_pats = 0
for dataset, pats in SELECTED.items():
    for pat in pats:
        ctxs  = all_contexts.get(dataset, {}).get(pat, [])
        valid = [c for c in ctxs if c.get('valid')]
        n     = len(valid)
        if n == 0: continue
        edfs_ok = all(c.get('path') and Path(c['path']).exists() and
                      c.get('inter_path') and Path(c['inter_path']).exists()
                      for c in valid)
        npz_n = sum(1 for k in range(1, n+1)
                    if npz_valido(SIGNAL_DIR / f'{dataset}__{pat}__ctx{k:03d}.npz'))
        if edfs_ok and npz_n == n:
            status = '✓ pronto'; ok_pats += 1
        elif npz_n >= 2:
            status = '⚠ parcial (LOSO ok)'; warn_pats += 1
        else:
            status = '✗ incompleto'; warn_pats += 1
        edf_tag = '✓' if edfs_ok else '⚠'
        npz_tag = '✓' if npz_n == n else f'⚠ {npz_n}/{n}'
        print(f'{dataset:<12} {pat:<14} {edf_tag:<8} {npz_tag:<10} {status}')
print('─'*60)
print(f'Pacientes prontos: {ok_pats} | com aviso: {warn_pats}')
print(f'.npz em disco: {len(list(SIGNAL_DIR.glob("*.npz")))}')


Dataset      Paciente       EDFs     npz        Status
────────────────────────────────────────────────────────────
CHBMIT       chb06          ✓        ✓          ✓ pronto
CHBMIT       chb10          ✓        ✓          ✓ pronto
CHBMIT       chb08          ✓        ✓          ✓ pronto
CHBMIT       chb14          ✓        ✓          ✓ pronto
CHBMIT       chb03          ✓        ✓          ✓ pronto
CHBMIT       chb05          ✓        ✓          ✓ pronto
CHBMIT       chb07          ✓        ✓          ✓ pronto
Siena        PN10           ✓        ✓          ✓ pronto
Siena        PN06           ✓        ✓          ✓ pronto
Siena        PN14           ✓        ✓          ✓ pronto
Siena        PN09           ✓        ✓          ✓ pronto
Siena        PN13           ✓        ✓          ✓ pronto
Siena        PN05           ✓        ✓          ✓ pronto
Siena        PN16           ✓        ✓          ✓ pronto
SeizeIT2     sub-073        ✓        ✓          ✓ pronto
SeizeIT2     sub-039        ✓

## 12. Exportação — mapa de contextos e manifesto

In [14]:
from datetime import datetime

# ── mapa_contextos_nb1.json ───────────────────────────────────────────────────
export = []
for dataset, pats in SELECTED.items():
    for pat in pats:
        if pat not in all_contexts.get(dataset, {}): continue
        ctx_id = 0
        for c in all_contexts[dataset][pat]:
            if not c['valid']: continue
            ctx_id += 1
            janelas = [w*60 for w in (10,20,30,40) if w*60 <= c['pre_max_disp_s']]
            export.append({
                'dataset': dataset, 'paciente': pat, 'contexto_id': ctx_id,
                'npz_path': str(SIGNAL_DIR / f'{dataset}__{pat}__ctx{ctx_id:03d}.npz'),
                'arquivo_crise': c['edf'],
                'onset_s': c['on_s'], 'offset_s': c['off_s'],
                'preictal': {
                    'arquivo': c['edf'],
                    'inicio_s': c['pre_inicio_s'], 'fim_s': c['pre_fim_s'],
                    'max_disp_s': c['pre_max_disp_s'],
                    'janelas_fixas_s': janelas,
                },
                'interictal': {
                    'arquivo': c['inter_arquivo'],
                    'inicio_s': c['inter_inicio_s'], 'fim_s': c['inter_fim_s'],
                    'duracao_s': c['inter_fim_s'] - c['inter_inicio_s'],
                    'fonte': c['inter_fonte'],
                },
            })

MAP_PATH = ROOT / 'mapa_contextos_nb1.json'
with open(MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(export, f, indent=2, ensure_ascii=False)
print(f'Exportados {len(export)} contextos → {MAP_PATH}')

# ── pipeline_manifest.json ────────────────────────────────────────────────────
MANIFEST_PATH = ROOT / 'pipeline_manifest.json'
manifest = {
    'generated_at': datetime.now().isoformat(),
    'parameters': {
        'GUARD_S': GUARD_S, 'MAX_PRE_S': MAX_PRE_S,
        'MIN_INTER_S': MIN_INTER_S, 'MIN_CONTEXTS': MIN_CONTEXTS,
        'MAX_INTER_SAVE_S': MAX_INTER_SAVE_S,
        'N_SELECT_PER_DS': N_SELECT_PER_DS,
        'EXCLUDE_PATIENTS': EXCLUDE_PATIENTS,
    },
    'selected_patients': {ds: list(pats) for ds, pats in SELECTED.items()},
    'contexts': [], 'summary': {},
}
n_saved = n_fail = 0
for ds, pats in SELECTED.items():
    for pat in pats:
        ctxs  = all_contexts.get(ds, {}).get(pat, [])
        valid = [c for c in ctxs if c.get('valid')]
        for k, c in enumerate(valid, 1):
            npz   = SIGNAL_DIR / f'{ds}__{pat}__ctx{k:03d}.npz'
            saved = npz_valido(npz)
            n_saved += saved; n_fail += not saved
            manifest['contexts'].append({
                'dataset': ds, 'paciente': pat, 'ctx_id': k,
                'edf_crise': c.get('edf',''), 'edf_inter': c.get('inter_arquivo',''),
                'onset_s': c.get('on_s'), 'inter_ini_s': c.get('inter_inicio_s'),
                'inter_fim_s': c.get('inter_fim_s'), 'pre_ini_s': c.get('pre_inicio_s'),
                'pre_fim_s': c.get('pre_fim_s'), 'npz_path': str(npz),
                'npz_saved': saved,
            })
manifest['summary'] = {
    'total_patients': sum(len(v) for v in SELECTED.values()),
    'total_contexts': len(manifest['contexts']),
    'contexts_saved': n_saved, 'contexts_failed': n_fail,
    'npz_em_disco': len(list(SIGNAL_DIR.glob('*.npz'))),
}
with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)

print(f'Manifesto salvo → {MANIFEST_PATH}')
print(f'  Pacientes : {manifest["summary"]["total_patients"]}')
print(f'  Contextos : {manifest["summary"]["total_contexts"]}')
print(f'  .npz ok   : {manifest["summary"]["contexts_saved"]}')
print(f'  Falhas    : {manifest["summary"]["contexts_failed"]}')

try:
    from google.colab import files
    files.download(str(MAP_PATH))
except ImportError:
    print(f'Arquivo local: {MAP_PATH.resolve()}')


Exportados 123 contextos → data\mapa_contextos_nb1.json
Manifesto salvo → data\pipeline_manifest.json
  Pacientes : 24
  Contextos : 123
  .npz ok   : 123
  Falhas    : 0
Arquivo local: D:\TCC\data\mapa_contextos_nb1.json
